# Article 1 — SAM 3.1 recorded-output runner

Run the same prepared video and concept prompt as the Grounding DINO + SAM 2.1 notebook, then download a directly comparable evidence bundle. This runner uses the official SAM 3.1 Object Multiplex predictor and records the exact upstream Git revision.

Before running: select a **GPU** runtime and add `HF_TOKEN` plus a read-only `GITHUB_TOKEN` under Colab **Secrets**. Confirm that the Hugging Face account behind `HF_TOKEN` can access the separate `facebook/sam3.1` repository. For the first comparison, use the identical video and semantic prompt in both notebooks without model-specific tuning.

The default configuration below is a **T4 smoke test**: both notebooks reduce the uploaded source to the first 3 seconds at 4 fps and at most 640 pixels wide. SAM 3.1 disables compilation and Flash Attention 3, falls back to FP16 when BF16 is unavailable, and tracks at most four objects. This establishes correctness on free Colab; it is not the final performance benchmark.

In [ ]:
import platform
import subprocess
import sys

print(f"Python: {platform.python_version()}")
assert sys.version_info >= (3, 12), "SAM 3.1 requires Python 3.12+"
gpu_report = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"GPU: {gpu_report}")
memory_mib = int(gpu_report.rsplit(",", 1)[1].strip().split()[0])
if memory_mib < 20 * 1024:
    print("WARNING: SAM 3.1 may run out of memory; shorten or downscale the video.")

In [ ]:
from base64 import b64encode
from pathlib import Path

from google.colab import userdata

ROOT = Path("/content")
GVI = ROOT / "grounded-visual-intelligence"
SAM3 = ROOT / "facebook-sam3"

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Add a read-only GITHUB_TOKEN in Colab Secrets")
basic = b64encode(f"x-access-token:{github_token}".encode()).decode()

if not GVI.exists():
    subprocess.run(
        [
            "git",
            "-c",
            f"http.extraHeader=Authorization: Basic {basic}",
            "clone",
            "--branch",
            "article/01-grounded-video-memory",
            "--depth",
            "1",
            "https://github.com/vlada22/grounded-visual-intelligence.git",
            str(GVI),
        ],
        check=True,
    )
if not SAM3.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/facebookresearch/sam3.git",
            str(SAM3),
        ],
        check=True,
    )

sam3_revision = subprocess.run(
    ["git", "-C", str(SAM3), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

# Guarded compatibility patch for facebookresearch/sam3#544. Remove this
# automatically once upstream filters init_state arguments itself.
base_predictor = SAM3 / "sam3/model/sam3_base_predictor.py"
source = base_predictor.read_text(encoding="utf-8")
needle = "        inference_state = self.model.init_state(**init_kwargs)\n"
if needle in source and "init_signature = inspect.signature" not in source:
    replacement = (
        "        import inspect\n\n"
        "        init_signature = inspect.signature(self.model.init_state)\n"
        "        init_kwargs = {\n"
        "            key: value for key, value in init_kwargs.items()\n"
        "            if key in init_signature.parameters\n"
        "        }\n"
        "        inference_state = self.model.init_state(**init_kwargs)\n"
    )
    base_predictor.write_text(source.replace(needle, replacement, 1), encoding="utf-8")
    print("Applied the guarded SAM 3.1 start-session compatibility patch.")

multiplex_predictor = SAM3 / "sam3/model/sam3_multiplex_video_predictor.py"
source = multiplex_predictor.read_text(encoding="utf-8")
needle = '        self.bf16_context = torch.autocast(device_type="cuda", dtype=torch.bfloat16)\n'
if needle in source:
    replacement = (
        "        amp_dtype = (\n"
        "            torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n"
        "        )\n"
        "        self.bf16_context = torch.autocast(\n"
        '            device_type="cuda", dtype=amp_dtype\n'
        "        )\n"
    )
    multiplex_predictor.write_text(source.replace(needle, replacement, 1), encoding="utf-8")
    print("Enabled the guarded FP16 fallback for pre-Ampere GPUs.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{GVI}[inference]",
        "-e",
        str(SAM3),
    ],
    check=True,
)
# Put the actual package root before /content so the outer clone directory cannot shadow it.
if str(SAM3) not in sys.path:
    sys.path.insert(0, str(SAM3))
sys.modules.pop("sam3", None)


In [ ]:
from huggingface_hub import HfApi, login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets after SAM 3.1 access is approved")
login(token=hf_token, add_to_git_credential=False)
try:
    model_info = HfApi().model_info("facebook/sam3.1", token=hf_token)
except Exception as error:
    raise RuntimeError(
        "HF_TOKEN cannot access facebook/sam3.1; verify the separate model approval"
    ) from error
print(f"Hugging Face access ready: {model_info.id}")

In [ ]:
from google.colab import files

print("Upload the same redistribution-safe source MP4 used for Grounded SAM 2.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one video")
original_video_path = ROOT / next(iter(uploaded))
video_path = ROOT / "article-01-comparison.mp4"
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        str(original_video_path),
        "-t",
        "3",
        "-vf",
        "fps=4,scale='min(640,iw)':-2:flags=lanczos",
        "-an",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        str(video_path),
    ],
    check=True,
)
print(f"Prepared comparison video: {video_path.name}")

In [ ]:
import json
from fractions import Fraction

probe = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height,avg_frame_rate,nb_frames,duration",
        "-of",
        "json",
        str(video_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
stream = json.loads(probe.stdout)["streams"][0]
fps = float(Fraction(stream["avg_frame_rate"]))
duration = float(stream.get("duration") or 0)
frame_count = int(stream.get("nb_frames") or round(duration * fps))
assert fps > 0 and frame_count > 0
video_info = {
    "width": int(stream["width"]),
    "height": int(stream["height"]),
    "fps": fps,
    "frame_count": frame_count,
}
video_info

In [ ]:
import time

import torch
from sam3.model_builder import build_sam3_multiplex_video_predictor

from gvi.adapters.sam3 import Sam3Adapter
from gvi.inference.sam3_worker import Sam3InferenceWorker, VideoDescriptor

PROMPT = "red toy car"  # Change this for the prepared scene.
SCENE_ID = "article-01-hero-sam31"
OUTPUT_DIR = ROOT / "gvi-artifacts" / SCENE_ID

descriptor = VideoDescriptor(
    scene_id=SCENE_ID,
    source_uri=f"prepared/{video_path.name}",
    **video_info,
)
predictor = build_sam3_multiplex_video_predictor(
    max_num_objects=4,
    use_fa3=False,
    use_rope_real=False,
    compile=False,
    warm_up=False,
)
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
recorded = Sam3InferenceWorker(predictor).run(
    video_path=video_path,
    video=descriptor,
    prompt=PROMPT,
    prompt_frame_index=0,
    output_dir=OUTPUT_DIR,
)
elapsed_s = time.perf_counter() - started
scene = Sam3Adapter().convert(recorded)
(OUTPUT_DIR / "scene.json").write_text(scene.model_dump_json(indent=2), encoding="utf-8")
metrics = {
    "pipeline": "sam-3.1-object-multiplex",
    "prompt": PROMPT,
    "sam3_git_revision": sam3_revision,
    "comparison_preset": {"seconds": 3, "fps": 4, "max_width": 640},
    "elapsed_s": elapsed_s,
    "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "tracked_frames": len(recorded.frames),
    "source_frames": descriptor.frame_count,
}
(OUTPUT_DIR / "run-metrics.json").write_text(json.dumps(metrics, indent=2))
metrics

In [ ]:
import shutil

archive = shutil.make_archive(str(ROOT / SCENE_ID), "zip", OUTPUT_DIR)
print(f"Artifact bundle: {archive}")
files.download(archive)

## Expected output

The ZIP is named `article-01-hero-sam31.zip`. It contains `sam3-output.json`, the model-independent `scene.json`, `run-metrics.json`, and one portable RLE mask per object observation. Keep the source video, tokens, and gated weights outside Git; send only this ZIP for comparison with `article-01-hero-grounded-sam2.zip`.